# SPARC Example 02: Baryonic Decomposition

**EPS Research RAG Astrophysics Corpus — Unified HI Corpus v7.0**

SPARC is unique: it provides not just Vobs but the full baryonic decomposition —
Vgas, Vdisk, and Vbul at every radial point. This is full-fidelity data directly
from Lelli et al. (2016), not derived or estimated.

We compute the baryonic velocity using sign-preserving quadrature:

    Vbar = sqrt(Upsilon*Vdisk^2 + Upsilon_b*Vbul^2 + sign(Vgas)*Vgas^2)

Note: Vgas can be negative at inner radii where thermal pressure exceeds rotation.

**Important note on corpus fidelity:** The `rotation_curve_corpus_v7_flat.csv` and `rotation_curve_corpus_v7.json` are **full-fidelity** — not a summary or veneer. The CSV contains every kinematic parameter published by Lelli et al. (2016) including per-galaxy inclination, distance uncertainties, mass-to-light ratios, and rotation curve statistics. The JSON adds full per-ring data: Vobs, Vgas, Vdisk, Vbul, errV at every radial point. This is the complete published dataset in a single machine-readable file.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.19563417  
**Source:** Lelli, McGaugh & Schombert (2016), AJ 152, 157  
**Dependencies:** Python 3, numpy, matplotlib, csv (standard library only)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('rotation_curve_corpus_v7.json') as f:
    corpus = json.load(f)

g = next(g for g in corpus['galaxies'] if g['galaxy'] == 'DDO161')
d = g['data']

R     = np.array([p['Rad']   for p in d])
Vobs  = np.array([p['Vobs']  for p in d])
errV  = np.array([p['errV']  for p in d])
Vgas  = np.array([p['Vgas']  for p in d])
Vdisk = np.array([p['Vdisk'] for p in d])
Vbul  = np.array([p['Vbul']  for p in d])

# Sign-preserving quadrature at Upsilon=1
Vbar = np.where(Vgas < 0,
                -np.sqrt(Vgas**2 + Vdisk**2 + Vbul**2),
                 np.sqrt(Vgas**2 + Vdisk**2 + Vbul**2))

print(f"Vgas negative rows: {(Vgas < 0).sum()} (inner radii — thermal pressure)")
print(f"Mass discrepancy at R_max:")
print(f"  Vobs = {Vobs[-1]:.1f} km/s")
print(f"  Vbar = {Vbar[-1]:.1f} km/s")
print(f"  Outer gap (Vobs - Vbar) = {Vobs[-1] - Vbar[-1]:.1f} km/s")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(R, Vobs, yerr=errV, fmt='o', color='#1f77b4',
            capsize=4, markersize=6, label=r'$V_{\rm obs}$ (SPARC)', zorder=5)
ax.plot(R, Vbar,  's-', color='#d62728', linewidth=1.5,
        label=r'$V_{\rm bar}$ (quadrature, $\Upsilon=1$)')
ax.plot(R, Vgas,  '^--', color='#2ca02c', linewidth=1.2, alpha=0.8,
        label=r'$V_{\rm gas}$')
ax.plot(R, Vdisk, 'v--', color='#ff7f0e', linewidth=1.2, alpha=0.8,
        label=r'$V_{\rm disk}$')
ax.axhline(0, color='black', linewidth=0.7, alpha=0.4)
ax.set_xlabel('Radius (kpc)', fontsize=12)
ax.set_ylabel('Velocity (km/s)', fontsize=12)
ax.set_title('DDO161 — Full Baryonic Decomposition\n'
             'Full-fidelity SPARC data, EPS Research Corpus v7.0', fontsize=11)
ax.legend(fontsize=9)
ax.text(0.02, 0.95,
        'Vgas < 0 at inner radii\n= thermal pressure term\n(sign-preserving quadrature)',
        transform=ax.transAxes, va='top', fontsize=8,
        bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', alpha=0.85))
plt.tight_layout()
plt.savefig('ex02_ddo161_baryonic.png', dpi=150, bbox_inches='tight')
plt.show()